# Week 1: Engineering Data Ingestion

Project: Culvert Exceedance Risk Assessment  
Author: Brice Nelson, PE, MBA  
Notebook: 00_data_ingestion.ipynb  
Week 1: Engineering Data Ingestion

In [1]:
# This notebook is designed to demonstrate the process of data ingestion for the Culvert Exceedance Risk Assessment project. The goal is to collect, clean, and prepare data for analysis.

# import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Project Root Directory
PROJECT_ROOT = Path.cwd().parents[2]

# Data directory
RAW_DATA_DIR = PROJECT_ROOT / "data/raw/engineering/week_01"

# Processed data directory
PROCESSED_DATA_DIR = PROJECT_ROOT / "data/processed/engineering/week_01"

# Create directory if it does not exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Output file
OUTPUT_FILE = PROCESSED_DATA_DIR / "annual_peak_flows_clean.csv"

# File Name
FILE_NAME = "annual_peak_flows_dirty.csv"

# File Path
FILE_PATH = RAW_DATA_DIR / FILE_NAME

print(f"Working directory: {Path.cwd()}")
print(f"Project root:      {PROJECT_ROOT}")
print(f"Reading file:      {FILE_PATH}")
print(f"File exists:       {FILE_PATH.exists()}")

Working directory: /home/brice-nelson/Documents/computerScience/PROJECTS/brice_dev_lab/monte_carlo_drills/notebooks/engineering/week_01
Project root:      /home/brice-nelson/Documents/computerScience/PROJECTS/brice_dev_lab/monte_carlo_drills
Reading file:      /home/brice-nelson/Documents/computerScience/PROJECTS/brice_dev_lab/monte_carlo_drills/data/raw/engineering/week_01/annual_peak_flows_dirty.csv
File exists:       True


In [2]:
# Load the data
try:
    df = pd.read_csv(FILE_PATH)
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"File not found: {FILE_PATH}")

Data loaded successfully.


In [3]:
# Display the first few rows of the dataframe
df.head(5)

,Water Year,Peak Flow,Units,Status
0,2007,217.7,cfs,Final
1,2005,398.3,cfs,Final
2,2006,130.6,cfs,Final
3,2016,502.4,cfs,Final
4,2023,429.2,cfs,Final


In [4]:
# Check shape of the dataframe
print(f"Dataframe shape: {df.shape}")

Dataframe shape: (31, 4)


In [5]:
# Inspect columns

df.columns

Index(['Water Year', 'Peak Flow', 'Units', 'Status'], dtype='str')

In [6]:
# Rename columns to standardize
df.rename(columns={
    'Water Year': 'water_year',
    'Peak Flow': 'peak_flow',
    'Units': 'units',
    'Status': 'status'
},
         inplace=True
         )
print("Columns have been renamed successfully")

Columns have been renamed successfully


In [7]:
# Verify columns were renamed
df.columns

Index(['water_year', 'peak_flow', 'units', 'status'], dtype='str')

In [8]:
# Check dtypes
df.dtypes

water_year      int64
peak_flow     float64
units             str
status            str
dtype: object

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   water_year  31 non-null     int64  
 1   peak_flow   30 non-null     float64
 2   units       31 non-null     str    
 3   status      31 non-null     str    
dtypes: float64(1), int64(1), str(2)
memory usage: 1.1 KB


In [10]:
# Identify if there are any duplicates
df.duplicated()

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12     True
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
30    False
dtype: bool

In [11]:
# Investigate duplicate
df[df.duplicated(keep=False)]

,water_year,peak_flow,units,status
1,2005,398.3,cfs,Final
12,2005,398.3,cfs,Final


In [12]:
# Drop duplicates
df = df.drop_duplicates()

In [13]:
# Check for duplicates
df.duplicated().sum()

np.int64(0)

In [14]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

Missing values in each column:
water_year    0
peak_flow     1
units         0
status        0
dtype: int64


In [15]:
# Determine which entries are missing
df[df.isna().any(axis=1)]

,water_year,peak_flow,units,status
5,2001,NaN,cfs,Final


In [16]:
# How many rows have each status
df["status"].value_counts()

status
Final          29
Provisional     1
Name: count, dtype: int64

In [17]:
# How many peak_flow values are missing within each status group?
df.groupby("status")["peak_flow"].apply(lambda x: x.isna().sum())



status
Final          1
Provisional    0
Name: peak_flow, dtype: int64

# Status and Missing-Value Check

The dataset contains **29 Final records** and **1 Provisional record**.

A separate missing-value check found that:

* **1 Final record** has a missing `peak_flow`.
* **0 Provisional records** have a missing `peak_flow`.

This means the missing `peak_flow` is **not associated with the provisional record**. Instead, the 2001 record is marked `Final` despite having no peak-flow value, indicating a potential data-quality issue.

**Recommended cleaning action:** Verify the 2001 peak-flow value against the original data source if possible. If the value cannot be recovered, exclude the record from analyses requiring `peak_flow` rather than replacing it with zero or an arbitrary value. Document the exclusion so the cleaning decision remains traceable.


In [18]:
# Remove the 2001 record because peak_flow is missing
# and the value could not be recovered from the source data.
df = df.dropna(subset=["peak_flow"])

# Validate the value has been dropped
df["peak_flow"].isna().sum()

np.int64(0)

In [19]:
# Verification of shape after the data cleaning
df.shape

(29, 4)

In [20]:
# Descriptive Statistics
df.describe()

,water_year,peak_flow
count,29.000000,29.000000
mean,2010.827586,364.848276
std,8.771174,168.720321
min,1996.000000,-45.000000
25%,2004.000000,260.100000
50%,2011.000000,378.500000
75%,2018.000000,446.100000
max,2025.000000,877.200000


In [21]:
# Identify negative value
df[df["peak_flow"] < 0]

,water_year,peak_flow,units,status
23,2019,-45.0,cfs,Final


In [22]:
# exclude the negative value
df = df[df["peak_flow"] >= 0]

# Print the min. value
min_value = df["peak_flow"].min()
print("Min. Value = ", min_value)

# Validate the value has been excluded
neg_values = (df["peak_flow"] < 0).sum()
print("Sum of negative values = ", neg_values)


Min. Value =  130.6
Sum of negative values =  0


In [23]:
# Updated shape after cleaning data
df.shape

(28, 4)

In [24]:
# Updated Descriptive Statistics
df.describe()

,water_year,peak_flow
count,28.000000,28.000000
mean,2010.535714,379.485714
std,8.787542,151.912325
min,1996.000000,130.600000
25%,2003.750000,264.300000
50%,2010.500000,379.800000
75%,2017.250000,447.125000
max,2025.000000,877.200000


In [25]:
# Save cleaned dataframe
df.to_csv(OUTPUT_FILE, index=False)



In [26]:
# Verify output file was saved
if OUTPUT_FILE.exists():
    print(f"Cleaned data saved to: {OUTPUT_FILE}")
else:
    print("Output file failed to save")
    

Cleaned data saved to: /home/brice-nelson/Documents/computerScience/PROJECTS/brice_dev_lab/monte_carlo_drills/data/processed/engineering/week_01/annual_peak_flows_clean.csv
